In [ ]:
from pyspark.sql import functions as F

In [ ]:
cols_to_drop_initial = [
    "ClmProcedureCode_3","ClmProcedureCode_4","ClmProcedureCode_5","ClmProcedureCode_6",
    "ClmDiagnosisCode_10",
    "DiagnosisGroupCode",
    "ClmAdmitDiagnosisCode"
]

claims_df = claims_df.drop(*[c for c in cols_to_drop_initial if c in claims_df.columns])

In [ ]:
# PotentialFraud → 1/0
claims_df = claims_df.withColumn(
    "PotentialFraud",
    F.when(F.col("PotentialFraud") == "Yes", 1).otherwise(0)
)

# Chronic Conditions → 1/0
chronic_cols = [c for c in claims_df.columns if "ChronicCond_" in c]

for col in chronic_cols:
    claims_df = claims_df.withColumn(
        col,
        F.when(F.col(col) == 1, 1).otherwise(0)
    )

# Renal Disease
claims_df = claims_df.withColumn(
    "HasRenalDisease",
    F.when(F.col("RenalDiseaseIndicator") == "Y", 1).otherwise(0)
)

In [ ]:
# Fill AttendingPhysician ONLY for counting
claims_df = claims_df.fillna({"AttendingPhysician": "UNKNOWN"})

# Median imputation (inpatient only)
median_val = claims_df.filter(F.col("claim_type") == 1) \
    .approxQuantile("DeductibleAmtPaid", [0.5], 0.01)[0]

claims_df = claims_df.withColumn(
    "DeductibleAmtPaid",
    F.when((F.col("claim_type") == 1) & (F.col("DeductibleAmtPaid").isNull()), median_val)
     .otherwise(F.col("DeductibleAmtPaid"))
)

# Diagnosis primary
claims_df = claims_df.withColumn(
    "ClmDiagnosisCode_1",
    F.when(F.col("ClmDiagnosisCode_1").isNull(), "UNKNOWN")
     .otherwise(F.col("ClmDiagnosisCode_1"))
)

In [ ]:
claims_df = claims_df.withColumn(
    "PhysicianCount",
    sum(F.when(F.col(c).isNotNull(),1).otherwise(0)
        for c in ["AttendingPhysician","OperatingPhysician","OtherPhysician"])
)

claims_df = claims_df.withColumn(
    "HasOperatingPhysician",
    F.when(F.col("OperatingPhysician").isNotNull(),1).otherwise(0)
)

claims_df = claims_df.withColumn(
    "HasOtherPhysician",
    F.when(F.col("OtherPhysician").isNotNull(),1).otherwise(0)
)

In [ ]:
diag_cols = [f"ClmDiagnosisCode_{i}" for i in range(1,10)]

claims_df = claims_df.withColumn(
    "DiagnosisCodeCount",
    sum(F.when(F.col(c).isNotNull(),1).otherwise(0) for c in diag_cols if c in claims_df.columns)
)

claims_df = claims_df.withColumn(
    "HasPrimaryDiagnosis",
    F.when(F.col("ClmDiagnosisCode_1") != "UNKNOWN",1).otherwise(0)
)

claims_df = claims_df.withColumn(
    "ICD_Chapter",
    F.substring("ClmDiagnosisCode_1",1,3)
)

In [ ]:
proc_cols = [c for c in ["ClmProcedureCode_1","ClmProcedureCode_2"] if c in claims_df.columns]

claims_df = claims_df.withColumn(
    "ProcedureCodeCount",
    sum(F.when(F.col(c).isNotNull(),1).otherwise(0) for c in proc_cols)
)

In [ ]:
claims_df = claims_df.withColumn("ClaimDuration", F.datediff("ClaimEndDt","ClaimStartDt"))
claims_df = claims_df.withColumn("LengthOfStay", F.datediff("DischargeDt","AdmissionDt"))

claims_df = claims_df.withColumn("Age", F.datediff("ClaimStartDt","DOB")/365.25)

claims_df = claims_df.withColumn(
    "IsDeceased",
    F.when(F.col("DOD").isNotNull(),1).otherwise(0)
)

In [ ]:
claims_df = claims_df.withColumn(
    "ChronicConditionCount",
    sum(F.col(c) for c in chronic_cols)
)

claims_df = claims_df.withColumn(
    "MedicareCoverageMonths",
    F.col("NoOfMonths_PartACov") + F.col("NoOfMonths_PartBCov")
)

claims_df = claims_df.withColumn(
    "IPtoOPReimbursementRatio",
    F.col("IPAnnualReimbursementAmt") / (F.col("OPAnnualReimbursementAmt") + 1)
)

In [ ]:
provider_stats = claims_df.groupBy("Provider").agg(
    F.count("*").alias("ProviderClaimCount"),
    F.avg("InscClaimAmtReimbursed").alias("ProviderAvgClaimAmt"),
    F.stddev("InscClaimAmtReimbursed").alias("ProviderClaimAmtStdDev"),
    F.avg("DiagnosisCodeCount").alias("ProviderAvgDiagnosisCount"),
    F.avg(F.when(F.col("claim_type")==1, F.col("LengthOfStay"))).alias("ProviderAvgLOS")
)

claims_df = claims_df.join(provider_stats, on="Provider", how="left")

In [ ]:
claims_df = claims_df.withColumn(
    "ClaimAmtZScore",
    F.when(F.col("ProviderClaimAmtStdDev") == 0, 0)
     .otherwise(
         (F.col("InscClaimAmtReimbursed") - F.col("ProviderAvgClaimAmt")) /
         F.col("ProviderClaimAmtStdDev")
     )
)

In [ ]:
cols_to_drop_final = [
    "AttendingPhysician","OperatingPhysician","OtherPhysician",
    "DOB","DOD","RenalDiseaseIndicator",
    "ClaimID","BeneID","Provider"
]

claims_df = claims_df.drop(*[c for c in cols_to_drop_final if c in claims_df.columns])

In [ ]:
claims_df = claims_df.fillna(0)

claims_df = claims_df.fillna({
    "Gender": "Unknown",
    "Race": "Unknown",
    "State": "Unknown",
    "ICD_Chapter": "UNK"
})

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

categorical_cols = ["Gender","Race","State","ICD_Chapter"]

indexers = [
    StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep")
    for c in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[c+"_idx" for c in categorical_cols],
    outputCols=[c+"_vec" for c in categorical_cols]
)

In [ ]:
from pyspark.ml.feature import VectorAssembler

numeric_cols = [
    "InscClaimAmtReimbursed","DeductibleAmtPaid",
    "ClaimDuration","LengthOfStay","Age",
    "DiagnosisCodeCount","ProcedureCodeCount",
    "PhysicianCount","ChronicConditionCount",
    "MedicareCoverageMonths","IPtoOPReimbursementRatio",
    "ProviderClaimCount","ProviderAvgClaimAmt",
    "ProviderAvgDiagnosisCount","ProviderAvgLOS",
    "ClaimAmtZScore"
]

binary_cols = [
    "claim_type","IsDeceased","HasRenalDisease",
    "HasOperatingPhysician","HasOtherPhysician","HasPrimaryDiagnosis"
]

feature_cols = numeric_cols + binary_cols + [c+"_vec" for c in categorical_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

In [ ]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=indexers + [encoder, assembler])

model_pipeline = pipeline.fit(claims_df)

final_df = model_pipeline.transform(claims_df)

In [ ]:
final_df = final_df.select("features","PotentialFraud")

final_df.printSchema()

In [ ]:
final_df.write.mode("overwrite").parquet("../data/day2_final_dataset")

provider_stats.write.mode("overwrite").parquet("../data/provider_lookup")